In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
latent_dim = 2
epochs = 5
lr = 1e-3
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")
mnist = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
x_all = torch.tensor(mnist.data, dtype=torch.float32).view(-1, 1, 28, 28) / 255.0
y_all = torch.tensor(mnist.target.astype("int64"), dtype=torch.long)

x_train = x_all[:60000]
y_train = y_all[:60000]

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
)

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 400)
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)
        self.fc3 = nn.Linear(latent_dim, 400)
        self.fc4 = nn.Linear(400, 28 * 28)
    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    def split_latent(self, mu, logvar):
        sigma = torch.exp(0.5 * logvar)
        z = mu + sigma.pow(2)
        return z, sigma
    def decode(self, z):
        h = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))
    def forward(self, x):
        x = x.view(-1, 28 * 28)
        mu, logvar = self.encode(x)
        z, sigma = self.split_latent(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, sigma, z
vae = VAE(latent_dim=latent_dim).to(device)

In [ ]:
def deterministic_latent_loss(recon_x, x, mu, sigma, lambda_mu=1e-4, lambda_sigma=1e-4):
    x_flat = x.view(-1, 28 * 28)
    recon_loss = F.binary_cross_entropy(recon_x, x_flat, reduction="sum")
    reg_mu = torch.sum(mu.pow(2))
    reg_sigma = torch.sum((sigma.pow(2) - 1.0).pow(2))
    total = recon_loss + lambda_mu * reg_mu + lambda_sigma * reg_sigma
    return total, recon_loss, reg_mu, reg_sigma

optimizer = torch.optim.Adam(vae.parameters(), lr=lr)

vae.train()
for epoch in range(epochs):
    epoch_total = 0.0
    epoch_recon = 0.0
    epoch_mu = 0.0
    epoch_sigma = 0.0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=(device.type == "cuda"))
        optimizer.zero_grad(set_to_none=True)
        recon, mu, logvar, sigma, _ = vae(x)
        total, recon_loss, reg_mu, reg_sigma = deterministic_latent_loss(recon, x, mu, sigma)
        total.backward()
        optimizer.step()

        epoch_total += total.item()
        epoch_recon += recon_loss.item()
        epoch_mu += reg_mu.item()
        epoch_sigma += reg_sigma.item()

    n = len(train_loader.dataset)
    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"total: {epoch_total / n:.4f} | recon: {epoch_recon / n:.4f} | "
        f"mu_reg: {epoch_mu / n:.4f} | sigma_reg: {epoch_sigma / n:.4f}"
    )

vae.eval()
with torch.no_grad():
    x, _ = next(iter(train_loader))
    x = x.to(device, non_blocking=(device.type == "cuda"))
    recon, mu, logvar, sigma, z = vae(x)
    total, recon_loss, reg_mu, reg_sigma = deterministic_latent_loss(recon, x, mu, sigma)

b = x.size(0)
print(
    f"Final batch (per sample) | total: {total.item() / b:.4f} | "
    f"recon: {recon_loss.item() / b:.4f} | "
    f"mu_reg: {reg_mu.item() / b:.4f} | sigma_reg: {reg_sigma.item() / b:.4f}"
)

In [ ]:
vae.eval()
latents = []
count = 0

with torch.no_grad():
    for x, _ in train_loader:
        x = x.to(device, non_blocking=(device.type == "cuda"))
        _, _, _, _, z = vae(x)
        latents.append(z.cpu())
        count += z.size(0)
        if count >= 20000:
            break

z_train = torch.cat(latents, dim=0).numpy()
gmm = GaussianMixture(
    n_components=20,
    covariance_type="diag",
    reg_covar=1e-4,
    random_state=42,
    max_iter=300,
 )
gmm.fit(z_train)

with torch.no_grad():
    x_vis, _ = next(iter(train_loader))
    x_vis = x_vis.to(device, non_blocking=(device.type == "cuda"))
    recon_vis, _, _, _, _ = vae(x_vis)
    x_vis = x_vis[:8].cpu()
    recon_vis = recon_vis.view(-1, 1, 28, 28)[:8].cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x_vis[i, 0], cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_vis[i, 0], cmap="gray")
    axes[1, i].axis("off")
plt.suptitle("Top: original | Bottom: reconstruction")
plt.tight_layout()
plt.show()

z_gen, _ = gmm.sample(16)
z_gen = torch.tensor(z_gen, dtype=torch.float32, device=device)

with torch.no_grad():
    gen = vae.decode(z_gen).view(-1, 1, 28, 28).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(gen[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("Generated digits from fitted latent prior")
plt.tight_layout()
plt.show()